In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from GX_271.gilson_ethernet import GilsonEthernet
from GX_271.tray import Tray
from GX_271.rack import Rack_209, Rack_3dp
from GX_271.flow_logging import FlowLogger
from GX_271.dim import DIM

#--- Project imports from other folders---
from WPI_Syr_pump.Pump import AL1000

#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.103", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to an "experiment") ---
logger = FlowLogger()
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(slot = 1, name = "rack1", module = rack1, x_offset = 35.6, y_offset = 8.1)
tray.add_module(slot = 2, name = "rack2", module = rack2, x_offset = 165, y_offset = 7.2) # Simulates another rack - useful later



In [2]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

syringe_dia = 4.606    # mm
rate = 0.5            # mL/min
volume = 1.2           # mL
direction = "INF"     # Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)

Device is connected

--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL1.2
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---



In [55]:
g.go_to_vial("rack1", 1)

Moving to rack1 vial 1 at (35.60, 8.10) mm


(np.float64(35.6), np.float64(8.1))

In [54]:
g.move_z(44, allow_in_vial=True)

'Moved Z to 44. Result: Move Z: 2, Success'

In [42]:
g.send_command("Get XY Position")

'Get XY Position: 118.35, 17.013'

In [31]:
g.move_y(17.014)

'Moved Y to 17.014. Result: Move Y: 2, Success'

In [36]:
g.move_x(52.7)

'Moved X to 52.7. Result: Move X: 2, Success'